# Étape 6 : règles de nettoyage

Ce notebook applique aux données chargées des règles de nettoyage explicites, dans un ordre fixe, sans jamais modifier les fichiers bruts : il travaille sur une copie, et chaque ligne garde son lignage. Chaque modification est inscrite dans un journal des corrections (table, ligne source, colonne, règle, valeur avant et après). Une valeur que la règle ne sait pas corriger sans ambiguïté reste telle quelle et reste une anomalie. Les valeurs des colonnes sensibles ne sont jamais affichées.

Chaque mois : mettre à jour `PERIODE_FIN`, exécuter d'abord le notebook `02c_dictionnaire`, puis celui-ci en entier.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from frigeo import racine_projet
from frigeo.perimetre import decouvrir

# PARAMETRE A MODIFIER CHAQUE MOIS
# Dernier mois déposé dans data/raw, au format AAAA-MM
PERIODE_FIN = "2026-03"

In [ ]:
%%time
# CHARGEMENT

from frigeo.chargement import charger_tout
from frigeo.controles import charger_dictionnaire_enregistre
from frigeo.dictionnaire import charger_meta

DATA = racine_projet() / "data"
perimetre = decouvrir(DATA, PERIODE_FIN)
perimetre.verifier()
print(perimetre.resume())

tables = charger_tout(perimetre)
# Sans dictionnaire enregistré, le message renvoie au notebook 02c_dictionnaire.
dictionnaire = charger_dictionnaire_enregistre()
print(len(tables), "tables,", sum(len(c) for c in dictionnaire.values()), "colonnes au dictionnaire")
print("dictionnaire de la période", charger_meta()["periode_fin"], ", données jusqu'à", PERIODE_FIN)

In [ ]:
# REGLES DE NETTOYAGE

from frigeo.nettoyage import catalogue_nettoyage

# Règles appliquées, dans leur ordre, avec ce qu'elles corrigent et ce qu'elles ne
# corrigent jamais.
display(catalogue_nettoyage())

In [ ]:
%%time
# NETTOYAGE

from frigeo.nettoyage import ACTION_MODIFIEE, ACTION_RETIREE, nettoyer

# Les tables chargées ne sont pas modifiées : tables_nettoyees en est une copie
# corrigée. Le journal porte des valeurs du client, il n'est pas affiché.
# La date de coupure (fin de la période extraite) borne les années écrites sur deux chiffres.
tables_nettoyees, journal, bilan_nettoyage = nettoyer(
    tables, dictionnaire, date_coupure=perimetre.date_coupure
)
print(int((journal["action"] == ACTION_MODIFIEE).sum()), "cellules corrigées,",
      int((journal["action"] == ACTION_RETIREE).sum()), "lignes retirées,",
      int(bilan_nettoyage["non_corrigees"].sum()), "cellules ou lignes visées non corrigées")
display(bilan_nettoyage)

In [ ]:
# COLONNES SENSIBLES

from frigeo.confidentialite import charger_sensibilite, est_sensible
from frigeo.controles import VALEUR_MASQUEE
from frigeo.dictionnaire import est_masquee
from frigeo.doublons import charger_declarations_doublons

sensibles = charger_sensibilite()
declarations = charger_declarations_doublons()


def sensible(table, colonne):
    regles = dictionnaire[table][colonne]
    return (est_sensible(sensibles, table, colonne) or est_masquee(regles["valeurs"])
            or est_masquee(regles["format"]))


# Les valeurs ne figurent que sur les cellules modifiées (une ligne retirée n'en a pas).
modifiees = journal[journal["action"] == ACTION_MODIFIEE]
sur_sensible = [sensible(t, c) for t, c in zip(modifiees["table"], modifiees["colonne"])]
cle_sensible = [
    any(est_sensible(sensibles, t, c) for c in declarations.get(t, {}).get("cle", []))
    for t in journal["table"]
]
masquees = ((modifiees["valeur_avant"] == VALEUR_MASQUEE)
            & (modifiees["valeur_apres"] == VALEUR_MASQUEE))
print("corrections sur une colonne sensible", sum(sur_sensible),
      ", dont valeurs non masquées (0 attendu)", int((~masquees[sur_sensible]).sum()))
print("lignes du journal dont la clé a une colonne sensible", sum(cle_sensible),
      ", dont clés non masquées (0 attendu)",
      int((journal.loc[cle_sensible, "cle"] != VALEUR_MASQUEE).sum()))

In [ ]:
%%time
# ANOMALIES APRES NETTOYAGE

from frigeo.anomalies import (
    anomalies_non_masquees,
    bilan_controles,
    charger_qualification,
    qualifier,
    reunir_anomalies,
)
from frigeo.controles import charger_regles_complementaires, controler
from frigeo.doublons import charger_declarations_doublons, detecter_doublons
from frigeo.gestion import charger_regles_gestion, controler_gestion

# Les contrôles des étapes 3 et 4 et la table des anomalies de l'étape 5, relancés
# sur les tables nettoyées. Ce qui reste est ce que les règles automatiques n'ont
# pas résolu, et sert au classeur d'arbitrage. Décomptes seulement.
declarations = charger_declarations_doublons()
ecarts_apres, bilan_apres = controler(
    tables_nettoyees, dictionnaire, complementaires=charger_regles_complementaires()
)
doublons_apres, bilan_doublons_apres = detecter_doublons(tables_nettoyees, declarations)
ecarts_gestion_apres, bilan_gestion_apres = controler_gestion(
    tables_nettoyees, dictionnaire, charger_regles_gestion(), declarations, doublons_apres,
    date_coupure=perimetre.date_coupure,
)
anomalies_apres = reunir_anomalies(
    tables_nettoyees, ecarts_apres, doublons_apres, ecarts_gestion_apres,
    bilan_apres, bilan_doublons_apres, bilan_gestion_apres, declarations,
)
bilan_anomalies_apres = bilan_controles(
    bilan_apres, bilan_doublons_apres, bilan_gestion_apres, anomalies_apres, declarations
)
anomalies_apres, bilan_anomalies_apres, _ = qualifier(
    anomalies_apres, bilan_anomalies_apres, charger_qualification(), tables_nettoyees
)
print(len(anomalies_apres), "lignes dans la table des anomalies après nettoyage,",
      anomalies_apres["id_anomalie"].nunique(), "identifiants distincts")
display(anomalies_apres.groupby(["etape", "issue"], sort=False).size().rename("lignes").to_frame())
print(len(anomalies_non_masquees(anomalies_apres, declarations=declarations)),
      "lignes dont la valeur ou la clé sensible n'est pas masquée (0 attendu)")

In [ ]:
# EXPORT ARBITRAGE

from frigeo.decisions import (
    ETATS_A_ARBITRER,
    charger_decisions,
    chemin_arbitrage,
    chemin_decisions,
    elements_a_arbitrer,
    exporter_arbitrage,
    repere_arbitrage,
)

# Éléments que les règles automatiques ne savent pas trancher, soumis au métier dans
# un classeur Excel (data/decisions). Le tableau des éléments porte des valeurs du
# client, il n'est pas affiché.
decisions = charger_decisions()
nb_imports = len(decisions["meta"]["imports"])
print(len(decisions["decisions"]), "décisions enregistrées,", nb_imports, "imports, lus dans",
      chemin_decisions())
elements = elements_a_arbitrer(tables_nettoyees, anomalies_apres, decisions)
display(pd.crosstab(elements["sorte"], elements["etat_decision"])
        .rename_axis(index="sorte", columns="etat_decision"))
a_exporter = int(elements["etat_decision"].isin(ETATS_A_ARBITRER).sum())
chemin_classeur = chemin_arbitrage(PERIODE_FIN)
if a_exporter == 0:
    print("Aucun élément à arbitrer, pas de classeur.")
elif chemin_classeur.exists() and repere_arbitrage(chemin_classeur) == nb_imports:
    print("Le classeur", chemin_classeur.name, "attend son import (cellule suivante).")
elif chemin_classeur.exists():
    print("Le classeur", chemin_classeur.name, "est déjà importé ou périmé. Le supprimer",
          "(Explorateur Windows, dossier data\\decisions), puis relancer cette cellule.")
else:
    exporter_arbitrage(elements, PERIODE_FIN, nb_imports=nb_imports)
    print(a_exporter, "éléments écrits dans", chemin_classeur)

In [ ]:
# IMPORT ARBITRAGE

from frigeo.decisions import charger_decisions, ecrire_decisions, importer_arbitrage

# À lancer quand le métier a rempli le classeur, l'a enregistré puis fermé dans Excel.
# Tous les problèmes sont listés en une fois, et rien n'est enregistré s'il y en a.
# Les décisions seront appliquées aux tables nettoyées au morceau 6e2.
chemin_classeur = chemin_arbitrage(PERIODE_FIN)
nb_imports = len(charger_decisions()["meta"]["imports"])
if not chemin_classeur.exists():
    print("Aucun classeur à importer dans", chemin_classeur.parent)
elif repere_arbitrage(chemin_classeur) != nb_imports:
    print("Le classeur", chemin_classeur.name, "est déjà importé ou périmé, rien n'est lu.")
else:
    arbitrage = importer_arbitrage(chemin_classeur, dictionnaire, nb_imports=nb_imports)
    print(len(arbitrage.decisions), "décisions lues dans", chemin_classeur.name)
    if len(arbitrage.decisions):
        display(pd.crosstab(arbitrage.decisions["sorte"], arbitrage.decisions["decision"])
                .rename_axis(index="sorte", columns="decision"))
    print(len(arbitrage.avertissements), "avertissements")
    for message in arbitrage.avertissements:
        print("  ", message)
    fichier, entree = ecrire_decisions(arbitrage, PERIODE_FIN)
    if entree is None:
        print("Aucune décision, rien n'est enregistré (le classeur reste importable).")
    else:
        print(entree["decisions"], f"décisions enregistrées dans {fichier}, dont",
              entree["remplacees"], "remplacent une décision caduque")
        print("Supprimer le classeur importé (dossier data\\decisions) avant le prochain export.")